In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# SellerPulse — Marketplace Seller Activation & Churn Analysis
*Starter notebook · built against the real Olist Brazilian E-Commerce schema*

**Before you run anything:** download the Olist dataset from Kaggle ("Olist Brazilian E-Commerce Public Dataset"), unzip it, and put the CSV files in the same folder as this notebook. Then run the cells top to bottom.

This notebook is structured in 4 phases matching your project brief:
1. **Build the seller base + lifecycle funnel + cohorts**
2. **Dropoff analysis + seller-health score**
3. **Export clean files for the dashboard**
4. **Notes for the PRD**

Wherever there's a *design decision* (a place where you, the PM, made a choice), it's flagged with `# DECISION:` — those are the spots you must be able to defend in an interview.

## Phase 0 — Setup & load

`DATA_DIR = "."` means "look in this folder." If your CSVs are elsewhere, change the path.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

DATA_DIR = Path("/content/drive/MyDrive/SellerPulse")

files = {
    "orders":     "olist_orders_dataset.csv",
    "items":      "olist_order_items_dataset.csv",
    "sellers":    "olist_sellers_dataset.csv",
    "reviews":    "olist_order_reviews_dataset.csv",
    "products":   "olist_products_dataset.csv",
    "cat_trans":  "product_category_name_translation.csv",
}

# Quick check that the files are present
missing = [f for f in files.values() if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        "Missing files: " + ", ".join(missing) +
        "\nDownload the Olist dataset from Kaggle and put the CSVs in this folder."
    )
print("All files found.")

All files found.


In [3]:
orders   = pd.read_csv(DATA_DIR / files["orders"])
items    = pd.read_csv(DATA_DIR / files["items"])
sellers  = pd.read_csv(DATA_DIR / files["sellers"])
reviews  = pd.read_csv(DATA_DIR / files["reviews"])
products = pd.read_csv(DATA_DIR / files["products"])
cat      = pd.read_csv(DATA_DIR / files["cat_trans"])

# Parse the date columns we'll use
date_cols = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

print("orders:", orders.shape, "| items:", items.shape, "| sellers:", sellers.shape)
orders.head(2)

orders: (99441, 8) | items: (112650, 7) | sellers: (3095, 4)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13


## Phase 1 — Build the seller base, lifecycle funnel & cohorts

### 1a. One row per (seller, order) = a "sale event"
Olist's `order_items` has one row per item, so a single order with 3 items shows up 3 times for a seller. We collapse to **distinct orders per seller**, because *that's* what "a sale" means for activation. We attach the order's purchase timestamp.

`# DECISION:` a "sale" = a distinct order, not a line item.

In [4]:
# Join item lines to their order's purchase timestamp, keep delivered/valid orders
valid_orders = orders[orders["order_status"] != "unavailable"].copy()

sales = (
    items[["order_id", "seller_id", "price"]]
    .merge(valid_orders[["order_id", "order_purchase_timestamp"]], on="order_id", how="inner")
)

# Collapse to one row per (seller, order): a sale event with its date and value
sale_events = (
    sales.groupby(["seller_id", "order_id"])
    .agg(sale_date=("order_purchase_timestamp", "min"),
         order_value=("price", "sum"))
    .reset_index()
    .dropna(subset=["sale_date"])
    .sort_values(["seller_id", "sale_date"])
)

# Rank each seller's sales: 1 = first sale, 2 = second sale, ...
sale_events["sale_rank"] = sale_events.groupby("seller_id").cumcount() + 1
# Days since that seller's FIRST sale
sale_events["first_sale_date"] = sale_events.groupby("seller_id")["sale_date"].transform("min")
sale_events["days_since_first"] = (sale_events["sale_date"] - sale_events["first_sale_date"]).dt.days

print("Total sale events:", len(sale_events), "| Unique sellers:", sale_events["seller_id"].nunique())
sale_events.head()

Total sale events: 100003 | Unique sellers: 3092


,seller_id,order_id,sale_date,order_value,sale_rank,first_sale_date,days_since_first
2,0015a82c2db000af6aaaf3ae2ecb0532,d455a8cb295653b55abda06d434ab492,2017-09-26 22:17:05,895.0,1,2017-09-26 22:17:05,0
1,0015a82c2db000af6aaaf3ae2ecb0532,9dc8d1a6f16f1b89874c29c9d8d30447,2017-10-12 13:33:22,895.0,2,2017-09-26 22:17:05,15
0,0015a82c2db000af6aaaf3ae2ecb0532,7f39ba4c9052be115350065d07583cac,2017-10-18 08:16:34,895.0,3,2017-09-26 22:17:05,21
56,001cca7ae9ae17fb1caed9dfb1094831,3c655487f0c8e34cde2c7b67de8f08cc,2017-02-04 19:06:04,99.9,1,2017-02-04 19:06:04,0
192,001cca7ae9ae17fb1caed9dfb1094831,eb188a175542057d90b3ca5628b7b5a0,2017-02-18 23:26:24,499.5,2,2017-02-04 19:06:04,14


### 1b. Handle observation censoring (an important, interview-worthy subtlety)
A seller whose first sale was 10 days before the dataset ends *cannot* show 90-day retention — they simply haven't had the time. Counting them as "churned" would be wrong. So for the 90-day funnel we only include sellers who had **at least 90 days of possible observation**.

`# DECISION:` exclude right-censored sellers from the 90-day funnel so retention isn't understated.

In [5]:
dataset_end = sale_events["sale_date"].max()
print("Dataset ends:", dataset_end.date())

seller_first = (
    sale_events.groupby("seller_id")
    .agg(first_sale_date=("first_sale_date", "first"),
         last_sale_date=("sale_date", "max"),
         total_sales=("order_id", "nunique"),
         total_gmv=("order_value", "sum"))
    .reset_index()
)
seller_first["observable_days"] = (dataset_end - seller_first["first_sale_date"]).dt.days

# Sellers with a fair shot at a 90-day window
cohort_base = seller_first[seller_first["observable_days"] >= 90].copy()
print(f"Sellers total: {len(seller_first)} | with >=90d observation window: {len(cohort_base)}")

Dataset ends: 2018-09-03
Sellers total: 3092 | with >=90d observation window: 2574


### 1c. The activation funnel
Stages:
- **Onboarded** — made a first sale (everyone in the base)
- **Reached 2nd sale within 30 days** — the key activation moment
- **Reached 5th sale** (within observation)
- **Active in days 30–90** — made any sale in that window (true retention, not just early burst)

`# DECISION:` the headline churn metric = *did not reach a 2nd sale within 30 days*. That matches the resume bullet and is the clearest early-churn signal.

In [6]:
def reached(seller_id, min_rank=2, within_days=None):
    s = sale_events[sale_events["seller_id"] == seller_id]
    s = s[s["sale_rank"] >= min_rank]
    if within_days is not None:
        s = s[s["days_since_first"] <= within_days]
    return len(s) > 0

# Vectorised versions (faster than looping)
g = sale_events.groupby("seller_id")

second_within_30 = g.apply(
    lambda s: ((s["sale_rank"] >= 2) & (s["days_since_first"] <= 30)).any()
).rename("second_sale_30d")

fifth_ever = g.apply(lambda s: (s["sale_rank"] >= 5).any()).rename("fifth_sale")

active_30_90 = g.apply(
    lambda s: ((s["days_since_first"] > 30) & (s["days_since_first"] <= 90)).any()
).rename("active_30_90")

funnel_flags = pd.concat([second_within_30, fifth_ever, active_30_90], axis=1).reset_index()
cohort_base = cohort_base.merge(funnel_flags, on="seller_id", how="left")

n = len(cohort_base)
funnel = pd.DataFrame({
    "stage": ["Onboarded (1st sale)", "2nd sale within 30d", "Active in days 30-90", "Reached 5th sale"],
    "sellers": [
        n,
        int(cohort_base["second_sale_30d"].sum()),
        int(cohort_base["active_30_90"].sum()),
        int(cohort_base["fifth_sale"].sum()),
    ],
})
funnel["pct_of_onboarded"] = (funnel["sellers"] / n * 100).round(1)
print(funnel.to_string(index=False))

churn_before_2nd = round((1 - cohort_base["second_sale_30d"].mean()) * 100, 1)
print(f"\n>>> HEADLINE: {churn_before_2nd}% of new sellers churn before a 2nd sale within 30 days.")

/tmp/ipykernel_651/2513651409.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  second_within_30 = g.apply(
/tmp/ipykernel_651/2513651409.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fifth_ever = g.apply(lambda s: (s["sale_rank"] >= 5).any()).rename("fifth_sale")


               stage  sellers  pct_of_onboarded
Onboarded (1st sale)     2574             100.0
 2nd sale within 30d     1532              59.5
Active in days 30-90     1657              64.4
    Reached 5th sale     1656              64.3

>>> HEADLINE: 40.5% of new sellers churn before a 2nd sale within 30 days.


/tmp/ipykernel_651/2513651409.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  active_30_90 = g.apply(


### 1d. Cohort retention (by month of first sale)
Group sellers by the month they joined, then show what fraction hit each milestone. This is the cohort table you'll visualise in the dashboard.

In [7]:
cohort_base["cohort_month"] = cohort_base["first_sale_date"].dt.to_period("M").astype(str)

cohort_table = (
    cohort_base.groupby("cohort_month")
    .agg(sellers=("seller_id", "nunique"),
         pct_2nd_30d=("second_sale_30d", "mean"),
         pct_active_30_90=("active_30_90", "mean"),
         pct_5th=("fifth_sale", "mean"))
    .reset_index()
)
for c in ["pct_2nd_30d", "pct_active_30_90", "pct_5th"]:
    cohort_table[c] = (cohort_table[c] * 100).round(1)

cohort_table.head(12)

,cohort_month,sellers,pct_2nd_30d,pct_active_30_90,pct_5th
0,2016-09,3,66.7,33.3,100.0
1,2016-10,136,41.2,1.5,78.7
2,2016-12,1,100.0,100.0,100.0
3,2017-01,152,72.4,77.0,76.3
4,2017-02,228,63.2,68.9,68.4
5,2017-03,173,64.7,67.6,67.1
6,2017-04,116,51.7,69.0,67.2
7,2017-05,124,57.3,64.5,59.7
8,2017-06,76,46.1,53.9,57.9
9,2017-07,116,65.5,75.0,74.1


## Phase 2 — Dropoff drivers & the seller-health score

### 2a. Top dropoff drivers by category
Which product categories have the worst early-churn? Attach each seller's **primary category** (the category they sell most in), then compute churn rate per category.

In [8]:
# Map each product to its (English) category
prod_cat = products[["product_id", "product_category_name"]].merge(cat, on="product_category_name", how="left")
prod_cat["category"] = prod_cat["product_category_name_english"].fillna(prod_cat["product_category_name"]).fillna("unknown")

# Seller's primary category = the category of most of their item lines
item_cat = items[["seller_id", "product_id"]].merge(prod_cat[["product_id", "category"]], on="product_id", how="left")
primary_cat = (
    item_cat.groupby(["seller_id", "category"]).size().reset_index(name="n")
    .sort_values(["seller_id", "n"], ascending=[True, False])
    .drop_duplicates("seller_id")[["seller_id", "category"]]
)
cohort_base = cohort_base.merge(primary_cat, on="seller_id", how="left")

# Churn rate by category (only categories with enough sellers to be meaningful)
# DECISION: require >=30 sellers per category so a tiny category can't top the list on noise.
cat_churn = (
    cohort_base.groupby("category")
    .agg(sellers=("seller_id", "nunique"),
         churn_before_2nd=("second_sale_30d", lambda x: (1 - x.mean()) * 100))
    .reset_index()
)
cat_churn = cat_churn[cat_churn["sellers"] >= 30].sort_values("churn_before_2nd", ascending=False)
cat_churn["churn_before_2nd"] = cat_churn["churn_before_2nd"].round(1)
print("Top 3 dropoff drivers (highest early churn):")
print(cat_churn.head(3).to_string(index=False))
cat_churn.head(10)

Top 3 dropoff drivers (highest early churn):
              category  sellers  churn_before_2nd
books_general_interest       36              58.3
   musical_instruments       35              51.4
          garden_tools       94              51.1


,category,sellers,churn_before_2nd
8,books_general_interest,36,58.3
52,musical_instruments,35,51.4
39,garden_tools,94,51.1
36,furniture_decor,197,49.2
65,unknown,96,49.0
16,construction_tools_construction,41,48.8
27,fashion_bags_accessories,51,47.1
5,auto,165,46.7
25,electronics,31,45.2
19,cool_stuff,71,42.3


### 2b. The seller-health score (0–100)
The PM centerpiece. Four components, each normalised to 0–100, then weighted.

| Component | Signal | Why it matters |
|---|---|---|
| Order velocity | sales per observable week | momentum / engagement |
| Delivery SLA | % delivered on/before estimate | seller reliability |
| Review | avg review score (1–5) | buyer satisfaction |
| Recency | days since last sale (inverted) | dormancy risk |

`# DECISION:` the **weights** below are your judgment call. The defaults weight reliability and momentum highest. Change them and have a reason — this is the #1 thing you'll be grilled on.

In [9]:
# ---- WEIGHTS: your design decision. Must sum to 1.0 ----
W_VELOCITY = 0.30
W_DELIVERY = 0.20
W_REVIEW   = 0.20
W_RECENCY  = 0.30
assert abs(W_VELOCITY + W_DELIVERY + W_REVIEW + W_RECENCY - 1.0) < 1e-9

# --- Component 1: order velocity (sales per observable week) ---
cohort_base["weeks_obs"] = (cohort_base["observable_days"] / 7).clip(lower=1)
cohort_base["velocity"] = cohort_base["total_sales"] / cohort_base["weeks_obs"]

# --- Component 2: delivery SLA (on-time rate per seller) ---
deliv = items[["order_id", "seller_id"]].merge(
    orders[["order_id", "order_delivered_customer_date", "order_estimated_delivery_date"]],
    on="order_id", how="left")
deliv = deliv.dropna(subset=["order_delivered_customer_date", "order_estimated_delivery_date"])
deliv["on_time"] = deliv["order_delivered_customer_date"] <= deliv["order_estimated_delivery_date"]
sla = deliv.groupby("seller_id")["on_time"].mean().rename("delivery_sla").reset_index()
cohort_base = cohort_base.merge(sla, on="seller_id", how="left")

# --- Component 3: avg review score per seller ---
rev = items[["order_id", "seller_id"]].merge(reviews[["order_id", "review_score"]], on="order_id", how="left")
rev = rev.dropna(subset=["review_score"])
avg_rev = rev.groupby("seller_id")["review_score"].mean().rename("avg_review").reset_index()
cohort_base = cohort_base.merge(avg_rev, on="seller_id", how="left")

# --- Component 4: recency (days since last sale, lower is better) ---
cohort_base["days_since_last"] = (dataset_end - cohort_base["last_sale_date"]).dt.days

# Fill gaps sensibly
cohort_base["delivery_sla"] = cohort_base["delivery_sla"].fillna(cohort_base["delivery_sla"].median())
cohort_base["avg_review"]   = cohort_base["avg_review"].fillna(cohort_base["avg_review"].median())

def norm(s, invert=False):
    """Min-max to 0-100; winsorise at 1st/99th pct so outliers don't dominate."""
    lo, hi = s.quantile(0.01), s.quantile(0.99)
    x = s.clip(lo, hi)
    out = (x - lo) / (hi - lo) * 100 if hi > lo else pd.Series(50, index=s.index)
    return 100 - out if invert else out

cohort_base["score_velocity"] = norm(cohort_base["velocity"])
cohort_base["score_delivery"] = cohort_base["delivery_sla"] * 100          # already 0-1
cohort_base["score_review"]   = (cohort_base["avg_review"] - 1) / 4 * 100   # 1-5 -> 0-100
cohort_base["score_recency"]  = norm(cohort_base["days_since_last"], invert=True)

cohort_base["health_score"] = (
    W_VELOCITY * cohort_base["score_velocity"] +
    W_DELIVERY * cohort_base["score_delivery"] +
    W_REVIEW   * cohort_base["score_review"] +
    W_RECENCY  * cohort_base["score_recency"]
).round(1)

print(cohort_base["health_score"].describe().round(1))
cohort_base[["seller_id", "category", "health_score",
             "score_velocity", "score_delivery", "score_review", "score_recency"]].head()

count    2574.0
mean       57.4
std        13.9
min         0.4
25%        49.0
50%        60.7
75%        66.8
max        95.5
Name: health_score, dtype: float64


,seller_id,category,health_score,score_velocity,score_delivery,score_review,score_recency
0,0015a82c2db000af6aaaf3ae2ecb0532,small_appliances,47.4,0.689395,100.000000,66.666667,46.270490
1,001cca7ae9ae17fb1caed9dfb1094831,garden_tools,71.1,33.690193,94.444444,72.563559,91.983216
2,002100f778ceb8431b7a1020ff7ab48f,furniture_decor,58.7,13.856499,83.333333,74.553571,76.461357
3,003554e2dce176b5555353e4f3555ac8,unknown,56.9,0.204561,100.000000,100.000000,56.163542
4,004c9cd9d87a3c30c522c48c4fc07416,bed_bath_table,66.0,26.213162,92.261905,78.323699,80.043325


### 2c. Flag at-risk sellers
Low health score = candidates for intervention. `# DECISION:` "at-risk" = bottom-quartile health score. Adjust the threshold and justify it.

In [10]:
threshold = cohort_base["health_score"].quantile(0.25)
cohort_base["at_risk"] = cohort_base["health_score"] <= threshold
print(f"At-risk threshold (bottom quartile health): {threshold:.1f}")
print(f"At-risk sellers: {int(cohort_base['at_risk'].sum())} of {len(cohort_base)} "
      f"({cohort_base['at_risk'].mean()*100:.1f}%)")

At-risk threshold (bottom quartile health): 49.0
At-risk sellers: 646 of 2574 (25.1%)


## Phase 3 — Export clean files for the dashboard
These CSVs load straight into Power BI (or Tableau). Build visuals on top:
- **funnel.csv** → funnel/bar chart of the activation stages
- **cohort_retention.csv** → matrix/heatmap, cohort month × milestone
- **category_churn.csv** → bar chart of worst categories
- **seller_health.csv** → score distribution, at-risk list, scatter of components

In [11]:
out = Path("/content/drive/MyDrive/SellerPulse/dashboard_exports")
out.mkdir(exist_ok=True)

funnel.to_csv(out / "funnel.csv", index=False)
cohort_table.to_csv(out / "cohort_retention.csv", index=False)
cat_churn.to_csv(out / "category_churn.csv", index=False)

seller_health = cohort_base[[
    "seller_id", "category", "cohort_month", "first_sale_date", "last_sale_date",
    "total_sales", "total_gmv", "velocity", "delivery_sla", "avg_review",
    "days_since_last", "health_score", "at_risk",
    "second_sale_30d", "active_30_90", "fifth_sale",
]].copy()
seller_health.to_csv(out / "seller_health.csv", index=False)

print("Exported to ./dashboard_exports/:")
for f in out.iterdir():
    print("  -", f.name)

Exported to ./dashboard_exports/:
  - funnel.csv
  - cohort_retention.csv
  - category_churn.csv
  - seller_health.csv


## Phase 4 — Notes for the PRD

Pull these straight into your PRD as evidence:

1. **Headline finding** — the `churn_before_2nd` %: this is your problem statement.
2. **Top 3 dropoff categories** — from `category_churn.csv`: these scope *where* the problem concentrates.
3. **At-risk segment size** — from the at-risk flag: this sizes the opportunity.
4. **Health-score weighting rationale** — write down *why* you chose your four weights.

Then propose **3 interventions** mapped to what the data shows (e.g. an early-seller onboarding nudge in days 0–30 for low-velocity sellers; delivery-SLA support for sellers with poor on-time rates; a category-specific activation playbook for your worst-churn category). Rank them with **RICE**.

Define your **North Star** (suggested: *% of new sellers reaching a 2nd sale within 30 days*) and **guardrails** (e.g. *don't let average delivery SLA or review score drop while pushing volume*).

---
### One honest caveat to keep in mind
Olist records a seller's *first sale*, not their *signup* — so "activation" here means "from first sale onward." Note this assumption explicitly in your PRD; naming the limitation of your own data is exactly the kind of rigor interviewers reward.